# Chapter 26 — Recommendations: Worked Retrofit Example

End-to-end reproduction of the Chapter 26 retrofit comparison. Three architectures on a common MLP-SCM substrate:

* **Original** TabICL-lite: linear attention with separate `W_Q, W_K`.
* **PSD retrofit**: shared `W_QK`; `B = W_QK^T W_QK` is PSD by construction.
* **General-symmetric retrofit** (SymGen): `B = W_U^T diag(D) W_U` with learnable signed diagonal `D`; symmetric but admits negative eigenvalues.

For each: train on the same prior with the same hyperparameters, apply Chapter 13's `inspect_attention` to read `alpha_A`, evaluate held-out MSE.

Cached audit: `book/data/cached_audits/retrofit_audit.json`.
Figures: `book/figures/fig_26_01_retrofit_arch.pdf` (architecture diagram), `book/figures/fig_26_02_retrofit_alpha_vs_loss.pdf` (results).


In [ ]:
import json, os, sys
import torch
import matplotlib.pyplot as plt

# Resolve repo root to import retrofit_runner from affinity/.
_p = os.getcwd()
while _p and not os.path.isdir(os.path.join(_p, 'affinity', 'book')):
    _p = os.path.dirname(_p)
sys.path.insert(0, os.path.join(_p, 'affinity'))

from retrofit_runner import (
    TabICLLite, TabICLLitePSD, TabICLLiteSymGen,
    _build_prior, _train, _evaluate, _energy_for_model,
    _param_count, _attn_param_count, N_EVAL_EPISODES,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)


## Train all three architectures on a common substrate

Same MLP-SCM prior, same hyperparameters, same seed. The only thing that varies is the attention parameterization.

In [ ]:
prior = _build_prior()
rows = []
for name, cls in [
    ('TabICL-lite (original)', TabICLLite),
    ('TabICL-lite-PSD', TabICLLitePSD),
    ('TabICL-lite-SymGen', TabICLLiteSymGen),
]:
    model = _train(cls, prior)
    energy = _energy_for_model(model)
    eval_result = _evaluate(model, prior, n_episodes=N_EVAL_EPISODES)
    rows.append({
        'architecture': name,
        'n_params_total': _param_count(model),
        'n_params_attention': _attn_param_count(model),
        **energy,
        **eval_result,
    })
    print(f"{name}: alpha_A={energy['mean_alpha_A']:.3f}  "
          f"MSE={eval_result['mean_mse']:.4f} +/- {eval_result['std_mse']:.4f}  "
          f"attn_params={_attn_param_count(model)}")


## Table 26.2 — Retrofit comparison

PSD and SymGen retrofits both drive `alpha_A` to zero by construction (the bilinear form is symmetric by parameterization). The held-out MSE between original and PSD is within seed noise; SymGen underperforms because row-normalization is unstable when scores can be negative (signed `D` admits negative eigenvalues).

In [ ]:
print(f"{'arch':<28s}  {'params':>9s}  {'attn':>7s}  {'alpha_A':>9s}  {'MSE':>10s}")
for r in rows:
    print(f"{r['architecture']:<28s}  {r['n_params_total']:>9d}  "
          f"{r['n_params_attention']:>7d}  "
          f"{r['mean_alpha_A']:>9.3f}  "
          f"{r['mean_mse']:>10.4f}")


## Figure 26.2 — `alpha_A` and held-out MSE

Generated and saved to `book/figures/fig_26_02_retrofit_alpha_vs_loss.pdf`.

In [ ]:
FIG = os.path.join(_p, 'affinity', 'book', 'figures',
                   'fig_26_02_retrofit_alpha_vs_loss.pdf')
short = ['Original', 'PSD retrofit', 'SymGen retrofit']
alpha_A = [r['mean_alpha_A'] for r in rows]
mse = [r['mean_mse'] for r in rows]
mse_std = [r['std_mse'] for r in rows]

fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.2))
ax0, ax1 = axes
ax0.bar(short, alpha_A, color=['#777777', '#3070b0', '#b04030'],
        edgecolor='black', linewidth=0.6)
ax0.set_ylabel(r'mean $\\alpha_A$ (asymmetric energy fraction)')
ax0.set_ylim(0, max(0.5, max(alpha_A) * 1.15))
ax0.axhline(0.5, color='black', linestyle=':', linewidth=0.6)
ax0.grid(True, axis='y', linestyle=':', alpha=0.4)

ax1.bar(short, mse, yerr=mse_std, capsize=4,
        color=['#777777', '#3070b0', '#b04030'],
        edgecolor='black', linewidth=0.6)
ax1.set_ylabel('held-out MSE (lower is better)')
ax1.grid(True, axis='y', linestyle=':', alpha=0.4)

plt.tight_layout()
plt.show()
print('saved', FIG)


## Reading

* The PSD retrofit cuts attention parameters by 25% (one shared `W_QK` instead of separate `W_Q, W_K`) at essentially no cost on this small-scale substrate (held-out MSE within seed noise of the original).
* The SymGen retrofit on a row-normalized linear-attention base underperforms: signed eigenvalues produce scores of mixed sign, which the row-normalization step does not handle well. SymGen is the safer default under softmax + MLP-head per Chapter 18 Recommendation 2; with linear attention + row-norm, PSD is the safer factorization.
* Both retrofits drive `alpha_A` to zero by construction, validating the kernel-method claim of Chapter 10 (the trained model carries no asymmetric energy because the parameterization forbids it).